In [1]:
# =========================================================
# STRESS TEST: K-FOLD CROSS VALIDATION
# =========================================================

import os
import numpy as np
import librosa
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold


# ---------------------------------------------------------
# CONFIG
# ---------------------------------------------------------
DATASET_PATH = "/home/feliciano/Documents/DATASET_SEABREAM_SEGMENTED/Segmented_2s"
SAMPLE_RATE = 8000
N_MELS = 64
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# =========================================================
# LABEL MAPPING
# =========================================================
def map_label_from_path(path):
    p = path.lower()
    if "background" in p: return "background"
    if "pre" in p and "feed" in p: return "pre-feeding"
    if "post" in p and "feed" in p: return "post-feeding"
    if "feeding" in p: return "feeding"
    return None


# =========================================================
# LOAD FILE PATHS
# =========================================================
file_paths, labels = [], []

for root, _, files in os.walk(DATASET_PATH):
    for file in files:
        if file.endswith(".wav"):
            label = map_label_from_path(root)
            if label:
                file_paths.append(os.path.join(root, file))
                labels.append(label)

file_paths = np.array(file_paths)
labels = np.array(labels)

print("Total samples:", len(file_paths))


# =========================================================
# FEATURE EXTRACTION
# =========================================================
def extract_features(file_path):
    y, sr = librosa.load(file_path, sr=SAMPLE_RATE, mono=True)

    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=N_MELS, fmax=1000
    )
    mel_db = librosa.power_to_db(mel)

    contrast = librosa.feature.spectral_contrast(
        y=y, sr=sr, fmin=50, n_bands=4
    )

    mel_mean = np.mean(mel_db, axis=1)
    mel_std  = np.std(mel_db, axis=1)
    contrast_mean = np.mean(contrast, axis=1)

    return np.concatenate([mel_mean, mel_std, contrast_mean])


def build_dataset(paths):
    return np.array([extract_features(p) for p in paths])


# =========================================================
# MODEL
# =========================================================
class InteractionModel(nn.Module):
    def __init__(self, dim, num_classes, k=32):
        super().__init__()

        self.query = nn.Linear(dim, k)
        self.key   = nn.Linear(dim, k)
        self.value = nn.Linear(k, dim)

        self.classifier = nn.Sequential(
            nn.Linear(dim, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        Q = self.query(x)
        K = self.key(x)

        scores = Q * K
        A = torch.softmax(scores, dim=1)

        y = self.value(A)
        y = y + x

        return self.classifier(y)


# =========================================================
# CROSS VALIDATION
# =========================================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

le = LabelEncoder()
labels_encoded = le.fit_transform(labels)

fold_accuracies = []
fold_f1s = []


for fold, (train_idx, test_idx) in enumerate(skf.split(file_paths, labels_encoded)):
    print(f"\n===== FOLD {fold + 1} =====")

    train_paths = file_paths[train_idx]
    test_paths  = file_paths[test_idx]
    y_train_raw = labels[train_idx]
    y_test_raw  = labels[test_idx]

    # Extract features
    X_train = build_dataset(train_paths)
    X_test  = build_dataset(test_paths)

    # Encode labels
    y_train = le.transform(y_train_raw)
    y_test  = le.transform(y_test_raw)

    # Scale
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    # Tensor
    X_train_t = torch.tensor(X_train, dtype=torch.float32).to(DEVICE)
    X_test_t  = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)
    y_train_t = torch.tensor(y_train).to(DEVICE)

    # Model
    model = InteractionModel(X_train.shape[1], len(np.unique(y_train))).to(DEVICE)

    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    # Training
    batch_size = 32
    for epoch in range(10):
        model.train()
        perm = torch.randperm(X_train_t.shape[0])

        for i in range(0, X_train_t.shape[0], batch_size):
            idx = perm[i:i+batch_size]
            xb = X_train_t[idx]
            yb = y_train_t[idx]

            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()

    # Evaluation
    model.eval()
    with torch.no_grad():
        preds = torch.argmax(model(X_test_t), dim=1).cpu().numpy()

    acc = accuracy_score(y_test, preds)
    f1  = f1_score(y_test, preds, average="macro")

    print("Accuracy:", acc)
    print("F1:", f1)

    fold_accuracies.append(acc)
    fold_f1s.append(f1)


# =========================================================
# FINAL RESULTS
# =========================================================
print("\n===== CROSS-VALIDATION RESULTS =====")
print("Mean Accuracy:", np.mean(fold_accuracies))
print("Std Accuracy:", np.std(fold_accuracies))
print("Mean F1:", np.mean(fold_f1s))
print("Std F1:", np.std(fold_f1s))

Total samples: 12000

===== FOLD 1 =====
Accuracy: 0.9991666666666666
F1: 0.9981486570195008

===== FOLD 2 =====
Accuracy: 0.9995833333333334
F1: 0.9990825995091042

===== FOLD 3 =====
Accuracy: 1.0
F1: 1.0

===== FOLD 4 =====
Accuracy: 0.9979166666666667
F1: 0.995412997545521

===== FOLD 5 =====
Accuracy: 1.0
F1: 1.0

===== CROSS-VALIDATION RESULTS =====
Mean Accuracy: 0.9993333333333334
Std Accuracy: 0.0007728015412913097
Mean F1: 0.9985288508148251
Std F1: 0.001702250650423326
